# Label Propagation

---

## Overview

**Label Propagation** is a *semi-supervised* learning method. it uses a small number of labeled examples to classify a large set of unlabeled points by spreading labels across a graph structure.

Unlabeled points are passed in with label $= -1$.

## Algorithm

1. **Build an affinity matrix** $W$ using the RBF kernel:
$$W_{ij} = \exp\left(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2\right)$$

2. **Row-normalize** to get the transition matrix:
$$T = D^{-1} W \quad \text{where } D = \text{diag}(\text{row sums of } W)$$

3. **Iterate:**
$$F \leftarrow T \cdot F$$
then reset labeled rows back to their true one-hot labels after each step (**clamping**).

4. Assign each unlabeled point to $\arg\max$ class in $F$.

---

**Dataset:** Synthetic. two well-separated clusters with only a few labeled points per cluster.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme()

from rice_ml.unsupervised_learning import LabelPropagation
from rice_ml.preprocess import StandardScaler

In [ ]:
# Generate two clusters with only 2 labeled points each
rng = np.random.default_rng(42)
X0 = rng.normal([-3, -3], 0.5, (50, 2))
X1 = rng.normal([3, 3], 0.5, (50, 2))
X = np.vstack([X0, X1])

# All unlabeled (-1) except 2 points per class
y = np.full(100, -1)
y[0], y[1]   = 0, 0
y[99], y[98] = 1, 1

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

n_labeled = (y != -1).sum()
print(f'Total samples: {len(X)}')
print(f'Labeled:       {n_labeled} ({100*n_labeled/len(X):.0f}%)')
print(f'Unlabeled:     {(y == -1).sum()} ({100*(y==-1).sum()/len(X):.0f}%)')

In [ ]:
# Visualize before propagation
plt.figure(figsize=(10, 8))
plt.scatter(X_scaled[y == -1, 0], X_scaled[y == -1, 1],
            c='lightgray', label='Unlabeled', alpha=0.6, s=40)
plt.scatter(X_scaled[y == 0, 0], X_scaled[y == 0, 1],
            c='steelblue', label='Labeled class 0', s=150, marker='*', zorder=5)
plt.scatter(X_scaled[y == 1, 0], X_scaled[y == 1, 1],
            c='salmon', label='Labeled class 1', s=150, marker='*', zorder=5)
plt.xlabel('Feature 1', fontsize=14)
plt.ylabel('Feature 2', fontsize=14)
plt.title('Before Label Propagation: Only 4 Labeled Points', fontsize=18)
plt.legend(fontsize=13)
plt.show()

## Run Label Propagation

In [ ]:
lp = LabelPropagation(gamma=2.0, n_iterations=500)
lp.fit(X_scaled, y)

labels = lp.predict()
print(f'Predicted labels: {dict(zip(*np.unique(labels, return_counts=True)))}')

In [ ]:
# Visualize after propagation
colors = np.where(labels == 0, 'steelblue', 'salmon')

plt.figure(figsize=(10, 8))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=colors, alpha=0.7, s=40)
plt.scatter(X_scaled[y != -1, 0], X_scaled[y != -1, 1],
            c='black', s=200, marker='*', zorder=5, label='Originally labeled')
plt.xlabel('Feature 1', fontsize=14)
plt.ylabel('Feature 2', fontsize=14)
plt.title('After Label Propagation: All Points Classified', fontsize=18)
plt.legend(fontsize=13)
plt.show()

In [ ]:
# Accuracy against true cluster membership
y_true = np.array([0]*50 + [1]*50)
from rice_ml.metrics import accuracy_score

# Account for possible label flip (propagation may assign 0 to cluster 1)
acc = max(
    accuracy_score(y_true, labels),
    accuracy_score(y_true, 1 - labels)
)
print(f'Accuracy vs true clusters: {acc:.4f}')

## Effect of gamma on Propagation

- **Large gamma** → tight RBF kernel → labels propagate only to very close neighbors
- **Small gamma** → wide kernel → labels spread further across the graph

In [ ]:
gamma_values = [0.1, 1.0, 5.0, 20.0]
fig, axes = plt.subplots(1, len(gamma_values), figsize=(18, 5))

for ax, gamma in zip(axes, gamma_values):
    lp_g = LabelPropagation(gamma=gamma, n_iterations=300)
    lp_g.fit(X_scaled, y)
    pred = lp_g.predict()
    c = np.where(pred == 0, 'steelblue', 'salmon')
    ax.scatter(X_scaled[:, 0], X_scaled[:, 1], c=c, alpha=0.7, s=20)
    ax.scatter(X_scaled[y != -1, 0], X_scaled[y != -1, 1],
               c='black', s=100, marker='*', zorder=5)
    ax.set_title(f'gamma={gamma}', fontsize=13)

plt.suptitle('Label Propagation: Effect of gamma', fontsize=16)
plt.tight_layout()
plt.show()

## Interpretation

- Label propagation leverages the **manifold assumption**: nearby points in feature space likely share the same label.
- Only **4 labeled points out of 100** are enough to classify the full dataset when clusters are well-separated.
- The RBF bandwidth $\gamma$ controls the neighborhood size. it should be tuned to match the scale of your data.
- **Clamping** ensures labeled points never change. they act as anchors that pull labels toward nearby unlabeled points.